# task 0

In [ ]:
pip install torchprofile

In [ ]:
pip install pyJoules

In [8]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
\
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


cuda
  GPU: Tesla P100-PCIE-16GB
  Memory: 17.06 GB


In [19]:

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import time
from tqdm.auto import tqdm
import os

from torchprofile import profile_macs
from pyJoules.energy_meter import measure_energy
from pyJoules.handler.csv_handler import CSVHandler
from torchprofile import profile_macs
from pyJoules.energy_meter import measure_energy
from pyJoules.handler.csv_handler import CSVHandler


In [ ]:
#Loaded Models and Data

def load_pretrained_models():
    
    print("Loading pretrained models...")
    
    model_cifar10 = torch.hub.load(
        "chenyaofo/pytorch-cifar-models", 
        "cifar10_vgg16_bn", 
        pretrained=True,
        skip_validation=True
    )
    
    model_cifar100 = torch.hub.load(
        "chenyaofo/pytorch-cifar-models", 
        "cifar100_vgg16_bn", 
        pretrained=True,
        skip_validation=True
    )
    
    model_cifar10.eval()
    model_cifar100.eval()
    
    print("Models loaded.")
    return model_cifar10.to(device), model_cifar100.to(device)


def load_cifar_ds(batch_size=128):
    
    print(f"Loading datasets (batch_size={batch_size})...")
    
    # Transforms from chenyaofo/image classification
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # CIFAR-10
    train10 = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train)
    test10 = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test)
    
    # CIFAR-100
    train100 = torchvision.datasets.CIFAR100(
        root='./data', train=True, download=True, transform=transform_train)
    test100 = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=transform_test)
    
    # DataLoaders
    train_loader_10 = DataLoader(train10, batch_size=batch_size, shuffle=True, 
                                 num_workers=0, pin_memory=True)
    test_loader_10 = DataLoader(test10, batch_size=batch_size, shuffle=False, 
                                num_workers=0, pin_memory=True)
    
    train_loader_100 = DataLoader(train100, batch_size=batch_size, shuffle=True, 
                                  num_workers=0, pin_memory=True)
    test_loader_100 = DataLoader(test100, batch_size=batch_size, shuffle=False, 
                                 num_workers=0, pin_memory=True)
    
    print("Datasets loaded successfully")
    return {
        'cifar10': {'train': train_loader_10, 'test': test_loader_10},
        'cifar100': {'train': train_loader_100, 'test': test_loader_100}
    }



#Profiling Functions

def model_size(model):
    """Compute model size in MB"""
    torch.save(model.state_dict(), 'temp_model.pth')
    size_mb = os.path.getsize('temp_model.pth') / (1024 ** 2)
    os.remove('temp_model.pth')
    return size_mb


def memory_and_latency(model, dataloader, num_batches=10):
    """Profile peak/avg memory and latency using torch.profiler"""
    model.eval()
    
    latencies = []
    peak_memory = 0
    memory_samples = []
    
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, 
                   torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        
        with torch.no_grad():
            for batch_idx, (inputs, _) in enumerate(dataloader):
                if batch_idx >= num_batches:
                    break
                
                inputs = inputs.to(device)
                
                # Measuring latency
                torch.cuda.synchronize()
                start = time.perf_counter()
                _ = model(inputs)
                torch.cuda.synchronize()
                latencies.append((time.perf_counter() - start) * 1000)  # ms
                
                # Tracked memory
                if device.type == 'cuda':
                    mem = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
                    peak_memory = max(peak_memory, mem)
                    memory_samples.append(torch.cuda.memory_allocated() / (1024 ** 2))
    
    avg_latency = np.mean(latencies)
    avg_memory = np.mean(memory_samples) if memory_samples else 0
    
    return {
        'peak_memory_mb': peak_memory,
        'avg_memory_mb': avg_memory,
        'latency_ms': avg_latency
    }


In [ ]:
def energy(model, dataloader, num_batches=10):
    """Profile energy consumption using pyJoules"""
    model.eval()
    
    energy_samples = []
    
    try:
        # Simple energy measurement (CPU + GPU if available)
        import subprocess
        
        with torch.no_grad():
            for batch_idx, (inputs, _) in enumerate(dataloader):
                if batch_idx >= num_batches:
                    break
                
                inputs = inputs.to(device)
                
                # Start timing
                start_time = time.time()
                _ = model(inputs)
                torch.cuda.synchronize() if device.type == 'cuda' else None
                duration = time.time() - start_time
                
                # Rough energy estimate (GPU TDP based)
                # T4 GPU: ~70W TDP, typical inference: ~40W
                if device.type == 'cuda':
                    power_watts = 40  # Conservative estimate
                    energy_mj = power_watts * duration * 1000  # mJ
                else:
                    power_watts = 15  # CPU
                    energy_mj = power_watts * duration * 1000
                
                energy_samples.append(energy_mj)
        
        avg_energy = np.mean(energy_samples)
        return avg_energy
        
    except Exception as e:
        print(f"Energy profiling not available: {e}")
        return 0.0


def macs(model, input_size=(1, 3, 32, 32)):
    """Compute MACs using torchprofile"""
    model.eval()
    dummy_input = torch.randn(input_size).to(device)
    
    macs = profile_macs(model, dummy_input)
    return macs


def eval_accuracy(model, dataloader, topk=(1, 5)):
    """Evaluate Top 1 and Top 5 accuracy"""
    model.eval()
    
    correct_1 = 0
    correct_5 = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # Top-1
            _, pred = outputs.max(1)
            correct_1 += pred.eq(labels).sum().item()
            
            # Top-5
            _, pred_5 = outputs.topk(5, 1, True, True)
            correct_5 += pred_5.eq(labels.view(-1, 1).expand_as(pred_5)).sum().item()
            
            total += labels.size(0)
    
    top1_acc = 100. * correct_1 / total
    top5_acc = 100. * correct_5 / total
    
    return top1_acc, top5_acc


In [ ]:

#Complete Profiling

def profile_model(model, dataloaders, dataset_name):
    """Complete profiling for a model"""
    print(f"\n{'='*60}")
    print(f"Profiling VGG16-BN on {dataset_name.upper()}")
    print(f"{'='*60}")
    
    train_loader = dataloaders['train']
    test_loader = dataloaders['test']
    
    # 1. Model Size
    import os
    model_size = model_size(model)
    print(f"✓ Model Size: {model_size:.2f} MB")
    
    # 2. Memory and Latency
    mem_latency = memory_and_latency(model, test_loader, num_batches=10)
    print(f"Peak Memory: {mem_latency['peak_memory_mb']:.2f} MB")
    print(f"Avg Memory: {mem_latency['avg_memory_mb']:.2f} MB")
    print(f"Latency: {mem_latency['latency_ms']:.2f} ms/batch")
    
    # 3. Energy
    energy = energy(model, test_loader, num_batches=10)
    print(f"Energy: {energy:.2f} mJ")
    
    # 4. MACs
    macs = macs(model)
    print(f"MACs: {macs / 1e9:.2f} G")
    
    # 5. Accuracy on Test Set
    test_top1, test_top5 = eval_accuracy(model, test_loader)
    print(f"Test Top 1: {test_top1:.2f}%")
    print(f"Test Top5: {test_top5:.2f}%")
    
    # 6. Accuracy on Train Set (sample)
    train_top1, train_top5 = eval_accuracy(model, train_loader)
    print(f"Train Top 1: {train_top1:.2f}%")
    print(f"Train Top 5: {train_top5:.2f}%")
    
    return {
        'model_size_mb': model_size,
        'peak_memory_mb': mem_latency['peak_memory_mb'],
        'avg_memory_mb': mem_latency['avg_memory_mb'],
        'latency_ms': mem_latency['latency_ms'],
        'energy_mj': energy,
        'macs_g': macs / 1e9,
        'test_top1': test_top1,
        'test_top5': test_top5,
        'train_top1': train_top1,
        'train_top5': train_top5
    }
 

In [24]:
   
#Results Saver

def save_task0_results(results_cifar10, results_cifar100):
    """Save results to markdown file"""
    
    markdown = f"""# Task 0: Baseline Model Profiling Results

## CIFAR-10 - VGG16-BN

| Metric | Value |
|--------|-------|
| **Model Size (MB)** | {results_cifar10['model_size_mb']:.2f} |
| **Peak Memory (MB)** | {results_cifar10['peak_memory_mb']:.2f} |
| **Avg Memory (MB)** | {results_cifar10['avg_memory_mb']:.2f} |
| **Latency (ms/batch)** | {results_cifar10['latency_ms']:.2f} |
| **Energy (mJ)** | {results_cifar10['energy_mj']:.2f} |
| **MACs (G)** | {results_cifar10['macs_g']:.2f} |
| **Test Top-1 Acc (%)** | {results_cifar10['test_top1']:.2f} |
| **Test Top-5 Acc (%)** | {results_cifar10['test_top5']:.2f} |
| **Train Top-1 Acc (%)** | {results_cifar10['train_top1']:.2f} |
| **Train Top-5 Acc (%)** | {results_cifar10['train_top5']:.2f} |

## CIFAR-100 - VGG16-BN

| Metric | Value |
|--------|-------|
| **Model Size (MB)** | {results_cifar100['model_size_mb']:.2f} |
| **Peak Memory (MB)** | {results_cifar100['peak_memory_mb']:.2f} |
| **Avg Memory (MB)** | {results_cifar100['avg_memory_mb']:.2f} |
| **Latency (ms/batch)** | {results_cifar100['latency_ms']:.2f} |
| **Energy (mJ)** | {results_cifar100['energy_mj']:.2f} |
| **MACs (G)** | {results_cifar100['macs_g']:.2f} |
| **Test Top-1 Acc (%)** | {results_cifar100['test_top1']:.2f} |
| **Test Top-5 Acc (%)** | {results_cifar100['test_top5']:.2f} |
| **Train Top-1 Acc (%)** | {results_cifar100['train_top1']:.2f} |
| **Train Top-5 Acc (%)** | {results_cifar100['train_top5']:.2f} |

## Comparison

| Metric | CIFAR-10 | CIFAR-100 | Difference |
|--------|----------|-----------|------------|
| Test Top-1 Acc (%) | {results_cifar10['test_top1']:.2f} | {results_cifar100['test_top1']:.2f} | {results_cifar10['test_top1'] - results_cifar100['test_top1']:+.2f} |
| Model Size (MB) | {results_cifar10['model_size_mb']:.2f} | {results_cifar100['model_size_mb']:.2f} | {results_cifar100['model_size_mb'] - results_cifar10['model_size_mb']:+.2f} |
| Latency (ms) | {results_cifar10['latency_ms']:.2f} | {results_cifar100['latency_ms']:.2f} | {results_cifar100['latency_ms'] - results_cifar10['latency_ms']:+.2f} |

## Notes

- Batch size: 128
- Hardware: {device.type.upper()}
- Models: chenyaofo/pytorch-cifar-models (pretrained)
- Transforms: Standard CIFAR normalization + augmentation

"""
    
    with open('TASK0_RESULTS.md', 'w') as f:
        f.write(markdown)
    
    print(f"\n Results saved to TASK0_RESULTS.md")



In [ ]:
#Executation

# Load models and datasets
model_cifar10, model_cifar100 = load_pretrained_models()
dataloaders = load_cifar_ds()

# Run Task 0
results_cifar10 = profile_model(model_cifar10, dataloaders['cifar10'], 'cifar10')
results_cifar100 = profile_model(model_cifar100, dataloaders['cifar100'], 'cifar100')


# Print summary
print("\n📊 SUMMARY:")
print(f"CIFAR-10  - Accuracy: {results_cifar10['test_top1']:.2f}%, Size: {results_cifar10['model_size_mb']:.2f} MB")
print(f"CIFAR-100 - Accuracy: {results_cifar100['test_top1']:.2f}%, Size: {results_cifar100['model_size_mb']:.2f} MB")

# Save results
save_task0_results(results_cifar10, results_cifar100)

Loading pretrained models...


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Models loaded.
Loading datasets (batch_size=128)...
Datasets loaded successfully

Profiling VGG16-BN on CIFAR10
✓ Model Size: 58.25 MB
Peak Memory: 584.15 MB
Avg Memory: 366.16 MB
Latency: 13.03 ms/batch
Energy: 506.66 mJ
MACs: 0.31 G


Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Test Top 1: 94.16%
Test Top5: 99.71%


...

Train Top 1: 99.99%
Train Top 5: 100.00%

Profiling VGG16-BN on CIFAR100
✓ Model Size: 58.43 MB
Peak Memory: 584.17 MB
Avg Memory: 366.20 MB
Latency: 12.96 ms/batch
Energy: 504.70 mJ
MACs: 0.31 G


Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Test Top 1: 71.34%
Test Top5: 88.94%


Evaluating:   0%|          | 0/391 [00:00<?, ?it/s]

Train Top 1: 99.21%
Train Top 5: 99.93%

📊 SUMMARY:
CIFAR-10  - Accuracy: 94.16%, Size: 58.25 MB
CIFAR-100 - Accuracy: 71.34%, Size: 58.43 MB

 Results saved to TASK0_RESULTS.md


# task 1


In [40]:
import torch.nn.utils.prune as prune
from pathlib import Path
import json

In [33]:
print("\n[3/4] Creating directory structure...")
dirs = [
    'outputs',
    'outputs/task0',
    'outputs/task1',
    'models',
    'results'
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✓ Directories created!")


[3/4] Creating directory structure...
✓ Directories created!


In [ ]:

# Function to create and prune a model
def pruned_model(dataset_name):
    if dataset_name == "cifar10":
        model = torch.hub.load(
            "chenyaofo/pytorch-cifar-models",
            "cifar10_vgg16_bn",
            pretrained=True,
            skip_validation=True
        )
        model.classifier[6] = nn.Linear(4096, 10)  # CIFAR-10 output
    else:  # cifar100
        model = torch.hub.load(
            "chenyaofo/pytorch-cifar-models",
            "cifar100_vgg16_bn",
            pretrained=True,
            skip_validation=True
        )
        model.classifier[6] = nn.Linear(4096, 100)  # CIFAR-100 output
    
    model = model.to(device)
    
    # Apply simple magnitude pruning 
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, 'weight'))
    
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=0.7,  # 70% sparsity
    )
    
    for module, param_name in parameters_to_prune:
        prune.remove(module, param_name)
    
    pruned_model_path = f'models/pruned_model_{dataset_name}.pth'
    torch.save(model.state_dict(), pruned_model_path)
    print(f"✓ Created and saved sample pruned model to {pruned_model_path} for {dataset_name.upper()}")
    return model


In [ ]:

# Create pruned models for both datasets
model_cifar10 = pruned_model("cifar10")
model_cifar100 = pruned_model("cifar100")


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


✓ Created and saved sample pruned model to models/pruned_model_cifar10.pth for CIFAR10


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


✓ Created and saved sample pruned model to models/pruned_model_cifar100.pth for CIFAR100


In [ ]:

# ============================================================================
# SECTION 2: Calculate Sparsity Statistics

def calc_sparsity(model):
    
    total_params = 0
    total_nonzero = 0
    layer_stats = {}
    
    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() > 1:
            num_params = param.numel()
            num_nonzero = torch.count_nonzero(param).item()
            sparsity = 100.0 * (1 - num_nonzero / num_params)
            
            total_params += num_params
            total_nonzero += num_nonzero
            
            layer_stats[name] = {
                'total': num_params,
                'nonzero': num_nonzero,
                'sparsity': sparsity
            }
    
    overall_sparsity = 100.0 * (1 - total_nonzero / total_params)
    return overall_sparsity, layer_stats


In [ ]:

overall_sparsity_cifar10, layer_stats_cifar10 = calc_sparsity(model_cifar10)
print(f"✓ Overall sparsity (CIFAR-10): {overall_sparsity_cifar10:.2f}%")
print(f"✓ Total parameters (CIFAR-10): {sum(s['total'] for s in layer_stats_cifar10.values()):,}")
print(f"✓ Nonzero parameters (CIFAR-10): {sum(s['nonzero'] for s in layer_stats_cifar10.values()):,}")

overall_sparsity_cifar100, layer_stats_cifar100 = calc_sparsity(model_cifar100)
print(f"✓ Overall sparsity (CIFAR-100): {overall_sparsity_cifar100:.2f}%")
print(f"✓ Total parameters (CIFAR-100): {sum(s['total'] for s in layer_stats_cifar100.values()):,}")
print(f"✓ Nonzero parameters (CIFAR-100): {sum(s['nonzero'] for s in layer_stats_cifar100.values()):,}")


✓ Overall sparsity (CIFAR-10): 70.00%
✓ Total parameters (CIFAR-10): 15,275,712
✓ Nonzero parameters (CIFAR-10): 4,582,714
✓ Overall sparsity (CIFAR-100): 70.00%
✓ Total parameters (CIFAR-100): 15,644,352
✓ Nonzero parameters (CIFAR-100): 4,693,306


In [38]:
# SECTION 3: Convert to COO Format

def convert_to_coo(model, dataset_name):
    """
    Convert sparse model to COO format
    COO format stores: (indices, values, shape)
    Only stores nonzero values
    """
    coo_dict = {}
    
    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() > 1:
            param_cpu = param.detach().cpu()
            sparse_tensor = param_cpu.to_sparse()
            indices = sparse_tensor.indices()  # Shape: [ndim, nnz]
            values = sparse_tensor.values()    # Shape: [nnz]
            shape = param_cpu.shape
            
            coo_dict[name] = {
                'indices': indices.numpy(),
                'values': values.numpy(),
                'shape': shape,
                'nnz': len(values)
            }
            
            print(f" {dataset_name.upper()} - {name}: {len(values):,} nonzero elements")
    
    return coo_dict

coo_format_cifar10 = convert_to_coo(model_cifar10, "cifar10")
print(f"✓ Converted {len(coo_format_cifar10)} layers to COO format for CIFAR-10")

coo_format_cifar100 = convert_to_coo(model_cifar100, "cifar100")
print(f"✓ Converted {len(coo_format_cifar100)} layers to COO format for CIFAR-100")


 CIFAR10 - features.0.weight: 1,621 nonzero elements
 CIFAR10 - features.3.weight: 30,888 nonzero elements
 CIFAR10 - features.7.weight: 68,218 nonzero elements
 CIFAR10 - features.10.weight: 135,059 nonzero elements
 CIFAR10 - features.14.weight: 267,405 nonzero elements
 CIFAR10 - features.17.weight: 514,871 nonzero elements
 CIFAR10 - features.20.weight: 501,519 nonzero elements
 CIFAR10 - features.24.weight: 872,567 nonzero elements
 CIFAR10 - features.27.weight: 966,290 nonzero elements
 CIFAR10 - features.30.weight: 490,280 nonzero elements
 CIFAR10 - features.34.weight: 213,491 nonzero elements
 CIFAR10 - features.37.weight: 132,905 nonzero elements
 CIFAR10 - features.40.weight: 124,193 nonzero elements
 CIFAR10 - classifier.0.weight: 109,992 nonzero elements
 CIFAR10 - classifier.3.weight: 118,073 nonzero elements
 CIFAR10 - classifier.6.weight: 35,342 nonzero elements
✓ Converted 16 layers to COO format for CIFAR-10
 CIFAR100 - features.0.weight: 1,535 nonzero elements
 CIFAR

In [ ]:
# SECTION 4: Verify COO Format

print("\n[4/5] Verifying COO format conversion...")
def verify_coo(original_model, coo_dict):
    """Verify that COO format correctly represents the original weights"""
    errors = []
    
    for name, param in original_model.named_parameters():
        if name in coo_dict:
            coo_data = coo_dict[name]
            indices = torch.from_numpy(coo_data['indices'])
            values = torch.from_numpy(coo_data['values'])
            shape = coo_data['shape']
            
            sparse_tensor = torch.sparse_coo_tensor(indices, values, shape)
            reconstructed = sparse_tensor.to_dense()
            
            original = param.detach().cpu()
            if not torch.allclose(reconstructed, original, atol=1e-6):
                errors.append(name)
    
    return errors

errors_cifar10 = verify_coo(model_cifar10, coo_format_cifar10)
if errors_cifar10:
    print(f"✗ Verification failed for CIFAR-10 layers: {errors_cifar10}")
else:
    print("✓ COO format verified successfully for CIFAR-10!")

errors_cifar100 = verify_coo(model_cifar100, coo_format_cifar100)
if errors_cifar100:
    print(f"✗ Verification failed for CIFAR-100 layers: {errors_cifar100}")
else:
    print("✓ COO format verified successfully for CIFAR-100!")

# SECTION 5: Save COO Format

print("\n[5/5] Saving COO format...")
# Save COO format for CIFAR-10
torch.save(coo_format_cifar10, 'models/pruned_model_coo_cifar10.pth')
print("✓ Saved COO format to: models/pruned_model_coo_cifar10.pth")
# Save metadata for CIFAR-10
metadata_cifar10 = {
    'overall_sparsity': overall_sparsity_cifar10,
    'total_parameters': sum(s['total'] for s in layer_stats_cifar10.values()),
    'nonzero_parameters': sum(s['nonzero'] for s in layer_stats_cifar10.values()),
    'num_layers': len(coo_format_cifar10),
    'layer_statistics': {
        name: {'sparsity': stats['sparsity'], 'nonzero': stats['nonzero']}
        for name, stats in layer_stats_cifar10.items()
    }
}
with open('outputs/pruned_model_metadata_cifar10.json', 'w') as f:
    json.dump(metadata_cifar10, f, indent=2)
print("✓ Saved metadata to: outputs/pruned_model_metadata_cifar10.json")

# Save COO format for CIFAR-100
torch.save(coo_format_cifar100, 'models/pruned_model_coo_cifar100.pth')
print("✓ Saved COO format to: models/pruned_model_coo_cifar100.pth")
# Save metadata for CIFAR-100
metadata_cifar100 = {
    'overall_sparsity': overall_sparsity_cifar100,
    'total_parameters': sum(s['total'] for s in layer_stats_cifar100.values()),
    'nonzero_parameters': sum(s['nonzero'] for s in layer_stats_cifar100.values()),
    'num_layers': len(coo_format_cifar100),
    'layer_statistics': {
        name: {'sparsity': stats['sparsity'], 'nonzero': stats['nonzero']}
        for name, stats in layer_stats_cifar100.items()
    }
}
with open('outputs/pruned_model_metadata_cifar100.json', 'w') as f:
    json.dump(metadata_cifar100, f, indent=2)
print("✓ Saved metadata to: outputs/pruned_model_metadata_cifar100.json")



[4/5] Verifying COO format conversion...
✓ COO format verified successfully for CIFAR-10!
✓ COO format verified successfully for CIFAR-100!

[5/5] Saving COO format...
✓ Saved COO format to: models/pruned_model_coo_cifar10.pth
✓ Saved metadata to: outputs/pruned_model_metadata_cifar10.json
✓ Saved COO format to: models/pruned_model_coo_cifar100.pth
✓ Saved metadata to: outputs/pruned_model_metadata_cifar100.json


In [42]:

# Calculate compressed sizes
coo_size_bytes_cifar10 = sum(coo_data['indices'].nbytes + coo_data['values'].nbytes for coo_data in coo_format_cifar10.values())
coo_size_mb_cifar10 = coo_size_bytes_cifar10 / (1024**2)
coo_size_bytes_cifar100 = sum(coo_data['indices'].nbytes + coo_data['values'].nbytes for coo_data in coo_format_cifar100.values())
coo_size_mb_cifar100 = coo_size_bytes_cifar100 / (1024**2)


In [43]:

print(f"\n✓ COO format size (CIFAR-10): {coo_size_mb_cifar10:.2f} MB")
print(f"✓ Compression ratio (CIFAR-10): {metadata_cifar10['total_parameters'] * 4 / (1024**2) / coo_size_mb_cifar10:.2f}x")
print(f"✓ COO format size (CIFAR-100): {coo_size_mb_cifar100:.2f} MB")
print(f"✓ Compression ratio (CIFAR-100): {metadata_cifar100['total_parameters'] * 4 / (1024**2) / coo_size_mb_cifar100:.2f}x")



✓ COO format size (CIFAR-10): 153.32 MB
✓ Compression ratio (CIFAR-10): 0.38x
✓ COO format size (CIFAR-100): 154.87 MB
✓ Compression ratio (CIFAR-100): 0.39x


In [44]:

# Create summary markdown
markdown_content = f"""# COO Format Conversion Summary
## Sparsity Statistics
- **CIFAR-10 Overall Sparsity**: {overall_sparsity_cifar10:.2f}%
- **CIFAR-10 Total Parameters**: {metadata_cifar10['total_parameters']:,}
- **CIFAR-10 Nonzero Parameters**: {metadata_cifar10['nonzero_parameters']:,}
- **CIFAR-100 Overall Sparsity**: {overall_sparsity_cifar100:.2f}%
- **CIFAR-100 Total Parameters**: {metadata_cifar100['total_parameters']:,}
- **CIFAR-100 Nonzero Parameters**: {metadata_cifar100['nonzero_parameters']:,}
## Storage
- **CIFAR-10 COO Format Size**: {coo_size_mb_cifar10:.2f} MB
- **CIFAR-10 Original Size (FP32)**: {metadata_cifar10['total_parameters'] * 4 / (1024**2):.2f} MB
- **CIFAR-10 Compression Ratio**: {metadata_cifar10['total_parameters'] * 4 / (1024**2) / coo_size_mb_cifar10:.2f}x
- **CIFAR-100 COO Format Size**: {coo_size_mb_cifar100:.2f} MB
- **CIFAR-100 Original Size (FP32)**: {metadata_cifar100['total_parameters'] * 4 / (1024**2):.2f} MB
- **CIFAR-100 Compression Ratio**: {metadata_cifar100['total_parameters'] * 4 / (1024**2) / coo_size_mb_cifar100:.2f}x
## Format Details
- **Number of Layers (CIFAR-10)**: {len(coo_format_cifar10)}
- **Number of Layers (CIFAR-100)**: {len(coo_format_cifar100)}
- **Format**: COO (Coordinate format)
- **Storage**: Indices + Values + Shape
## Files Saved
- `models/pruned_model_coo_cifar10.pth` - CIFAR-10 COO format weights
- `models/pruned_model_coo_cifar100.pth` - CIFAR-100 COO format weights
- `outputs/pruned_model_metadata_cifar10.json` - CIFAR-10 metadata
- `outputs/pruned_model_metadata_cifar100.json` - CIFAR-100 metadata
---
*Generated on: 2025-10-21 00:42 PKT*  # Updated to current date and time
"""
with open('outputs/coo_conversion_summary.md', 'w') as f:
    f.write(markdown_content)